## Objective

Understand how LangChain's TextLoader works, how it loads text files,
how it creates Document objects, how metadata is generated, and how
load() differs from lazy_load().

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader

In [ ]:
file_path = Path("../../data/raw/txt/azure_event_hubs_knowledge.txt")

In [ ]:
print(file_path)
print(file_path.exists())
print(file_path.resolve())

In [ ]:
loader = TextLoader(str(file_path))
type(loader)

In [ ]:
# print(loader)
print(loader.__dict__)

In [ ]:
# Inspect the constructor signature

import inspect
print(inspect.signature(TextLoader))

In [ ]:
documents = loader.load()

In [ ]:
print(type(documents))
print(len(documents))

In [ ]:
document = documents[0]
print(type(document))

In [ ]:
print(document.page_content)
print("Characters:", len(document.page_content))
print(document.metadata)
print("Source:", document.metadata.get("source"))

### Now understand lazy_load()

In [ ]:
documents = loader.load()

In [ ]:
documents_iterator = loader.lazy_load()

In [ ]:
print(type(documents_iterator))

                TextLoader
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
       load()             lazy_load()
          │                   │
          ▼                   ▼
    List[Document]         Iterator
          │                   │
          ▼                   ▼
 All documents          Documents produced
 available at once      as iteration occurs

In [ ]:
loader = TextLoader(
    str(file_path),
    encoding="utf-8"
)

for document in loader.lazy_load():
    print("Document type:", type(document))
    print("Characters:", len(document.page_content))
    print("Metadata:", document.metadata)

### Let's create a reusable inspection cell

In [22]:
def inspect_documents(documents):
    print(f"Number of documents: {len(documents)}")
    print()

    for index, document in enumerate(documents):
        print(f"Document {index + 1}")
        print("-" * 50)
        print(f"Type: {type(document)}")
        print(f"Characters: {len(document.page_content)}")
        print(f"Metadata: {document.metadata}")
        print()

In [23]:
loader = TextLoader(
    str(file_path),
    encoding="utf-8"
)

documents = loader.load()

inspect_documents(documents)

Number of documents: 1

Document 1
--------------------------------------------------
Type: <class 'langchain_core.documents.base.Document'>
Characters: 1308
Metadata: {'source': '..\\..\\data\\raw\\txt\\azure_event_hubs_knowledge.txt'}



### Stage 1.3 — Key Takeaways

## Key Takeaways

1. TextLoader is a LangChain document loader for text files.
2. Creating a TextLoader configures the source; it does not necessarily perform the loading immediately.
3. `load()` reads the source and returns a list of LangChain `Document` objects.
4. `lazy_load()` provides documents through an iterator, allowing incremental processing.
5. A TXT file commonly produces one Document with TextLoader.
6. The Document contains `page_content` and `metadata`.
7. Encoding is an important consideration when ingesting text files.
8. `autodetect_encoding` can be used when encoding may not be known in advance.
9. Missing or inaccessible source files can cause loader errors.
10. We intentionally keep loading separate from chunking so that each RAG stage is understood independently.

## What you should now understand
You should have this mental model:
                KNOWLEDGE SOURCE
                       │
                       ▼
                ┌────────────┐
                │   LOADER   │
                └──────┬─────┘
                       │
                       ▼
               List[Document]
                       │
              ┌────────┴────────┐
              ▼                 ▼
        page_content         metadata
              │
              ▼
       Next RAG Stage